# 0. Configure Project Path

In [32]:
import sys
from pathlib import Path

project_root = Path().resolve().parent

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

## 1. Import Libraries

In [33]:
import pandas as pd
import numpy as np

from src.preprocessing import build_decision_matrix

# 2. Load Dataset

In [34]:
df = pd.read_csv("../data/raw/projects.csv")

df.head()

,Project_ID,Project_Type,Team_Size,Project_Budget_USD,Estimated_Timeline_Months,Complexity_Score,Stakeholder_Count,Methodology_Used,Team_Experience_Level,Past_Similar_Projects,...,Industry_Volatility,Client_Experience_Level,Change_Control_Maturity,Risk_Management_Maturity,Team_Colocation,Documentation_Quality,Project_Start_Month,Current_Phase_Duration_Months,Seasonal_Risk_Factor,Risk_Level
0,PROJ_0001,Construction,32,1526276.55,32,9.70,16,Waterfall,Senior,3,...,Extreme,First-time,Basic,Basic,Fully Colocated,Good,10,5,1.0,High
1,PROJ_0002,Manufacturing,2,390790.15,9,2.72,9,Kanban,Mixed,0,...,Stable,Occasional,Advanced,Formal,Fully Remote,Poor,9,3,1.0,Low
2,PROJ_0003,Manufacturing,2,246674.76,6,2.04,7,Agile,Mixed,1,...,Stable,Regular,NaN,NaN,Hybrid,Good,5,1,1.0,Medium
3,PROJ_0004,IT,12,1427830.63,17,7.54,16,Scrum,Mixed,0,...,Extreme,Strategic,Formal,Basic,Hybrid,Basic,12,6,1.1,High
4,PROJ_0005,Construction,24,1696746.64,24,6.68,17,Hybrid,Junior,0,...,Moderate,Occasional,Basic,NaN,Partially Colocated,Basic,9,6,1.0,High


# 3. Define Decision Criteria

In [35]:
criteria_objectives = {
    "Project_Budget_USD": "min",
    "Estimated_Timeline_Months": "min",
    "Complexity_Score": "min",
    "Previous_Delivery_Success_Rate": "max",
    "Resource_Availability": "max",
    "Historical_Risk_Incidents": "min",
}

criteria_objectives

{'Project_Budget_USD': 'min',
 'Estimated_Timeline_Months': 'min',
 'Complexity_Score': 'min',
 'Previous_Delivery_Success_Rate': 'max',
 'Resource_Availability': 'max',
 'Historical_Risk_Incidents': 'min'}

# 4. Build the Decision Matrix

In [36]:
decision_matrix = build_decision_matrix(
    df,
    criteria_objectives
)

decision_matrix.head()

,Project_Budget_USD,Estimated_Timeline_Months,Complexity_Score,Previous_Delivery_Success_Rate,Resource_Availability,Historical_Risk_Incidents
0,1526276.55,32,9.70,0.80,0.98,2
1,390790.15,9,2.72,0.73,0.95,2
2,246674.76,6,2.04,0.91,0.79,2
3,1427830.63,17,7.54,0.71,0.52,1
4,1696746.64,24,6.68,0.83,0.58,1


# 5. Build the Pairwise Comparison Matrix

In [37]:
criteria_names = [
    "Project_Budget_USD",
    "Estimated_Timeline_Months",
    "Complexity_Score",
    "Previous_Delivery_Success_Rate",
    "Resource_Availability",
    "Historical_Risk_Incidents",
]

In [38]:
criteria_names = list(criteria_objectives.keys())

criteria_names

['Project_Budget_USD',
 'Estimated_Timeline_Months',
 'Complexity_Score',
 'Previous_Delivery_Success_Rate',
 'Resource_Availability',
 'Historical_Risk_Incidents']

In [39]:
pairwise_matrix = pd.DataFrame(
    [
        [1,   1/3, 1/5, 1/7, 1/3, 1/5],
        [3,   1,   1/3, 1/5, 1/2, 1/3],
        [5,   3,   1,   1/3, 3,   1],
        [7,   5,   3,   1,   5,   3],
        [3,   2,   1/3, 1/5, 1,   1/2],
        [5,   3,   1,   1/3, 2,   1],
    ],
    index=criteria_names,
    columns=criteria_names,
)

pairwise_matrix

,Project_Budget_USD,Estimated_Timeline_Months,Complexity_Score,Previous_Delivery_Success_Rate,Resource_Availability,Historical_Risk_Incidents
Project_Budget_USD,1,0.333333,0.200000,0.142857,0.333333,0.200000
Estimated_Timeline_Months,3,1.000000,0.333333,0.200000,0.500000,0.333333
Complexity_Score,5,3.000000,1.000000,0.333333,3.000000,1.000000
Previous_Delivery_Success_Rate,7,5.000000,3.000000,1.000000,5.000000,3.000000
Resource_Availability,3,2.000000,0.333333,0.200000,1.000000,0.500000
Historical_Risk_Incidents,5,3.000000,1.000000,0.333333,2.000000,1.000000


# 6. Normalize the Pairwise Comparison Matrix

In [40]:
normalized_pairwise = pairwise_matrix.div(
    pairwise_matrix.sum(axis=0),
    axis=1
)

normalized_pairwise

,Project_Budget_USD,Estimated_Timeline_Months,Complexity_Score,Previous_Delivery_Success_Rate,Resource_Availability,Historical_Risk_Incidents
Project_Budget_USD,0.041667,0.023256,0.034091,0.064655,0.028169,0.033149
Estimated_Timeline_Months,0.125000,0.069767,0.056818,0.090517,0.042254,0.055249
Complexity_Score,0.208333,0.209302,0.170455,0.150862,0.253521,0.165746
Previous_Delivery_Success_Rate,0.291667,0.348837,0.511364,0.452586,0.422535,0.497238
Resource_Availability,0.125000,0.139535,0.056818,0.090517,0.084507,0.082873
Historical_Risk_Incidents,0.208333,0.209302,0.170455,0.150862,0.169014,0.165746


In [41]:
normalized_pairwise.sum(axis=0)

Project_Budget_USD                1.0
Estimated_Timeline_Months         1.0
Complexity_Score                  1.0
Previous_Delivery_Success_Rate    1.0
Resource_Availability             1.0
Historical_Risk_Incidents         1.0
dtype: float64

# 7. Calculate the Priority Vector

In [42]:
priority_vector = normalized_pairwise.mean(axis=1)

priority_vector

Project_Budget_USD                0.037498
Estimated_Timeline_Months         0.073268
Complexity_Score                  0.193037
Previous_Delivery_Success_Rate    0.420704
Resource_Availability             0.096542
Historical_Risk_Incidents         0.178952
dtype: float64

# 8. Check the Consistency Ratio

In [43]:
weighted_sum = pairwise_matrix.dot(priority_vector)

weighted_sum

Project_Budget_USD                0.228599
Estimated_Timeline_Months         0.442169
Complexity_Score                  1.209140
Previous_Delivery_Success_Rate    2.648201
Resource_Availability             0.593533
Historical_Risk_Incidents         1.112598
dtype: float64

In [44]:
consistency_vector = weighted_sum / priority_vector

consistency_vector

Project_Budget_USD                6.096338
Estimated_Timeline_Months         6.034992
Complexity_Score                  6.263788
Previous_Delivery_Success_Rate    6.294683
Resource_Availability             6.147938
Historical_Risk_Incidents         6.217299
dtype: float64

In [45]:
lambda_max = consistency_vector.mean()

lambda_max

6.17583967793167

In [46]:
n = len(criteria_names)

ci = (lambda_max - n) / (n - 1)

ci

0.03516793558633395

In [47]:
ri = 1.24

cr = ci / ri

cr

0.028361238376075768

# 9. Normalize the Decision Matrix

In [48]:
normalized_decision = decision_matrix.copy()

for column, objective in criteria_objectives.items():

    minimum = decision_matrix[column].min()
    maximum = decision_matrix[column].max()

    if maximum == minimum:
        normalized_decision[column] = 1
        continue

    if objective == "max":

        normalized_decision[column] = (
            decision_matrix[column] - minimum
        ) / (maximum - minimum)

    else:

        normalized_decision[column] = (
            maximum - decision_matrix[column]
        ) / (maximum - minimum)

normalized_decision.head()

,Project_Budget_USD,Estimated_Timeline_Months,Complexity_Score,Previous_Delivery_Success_Rate,Resource_Availability,Historical_Risk_Incidents
0,0.621246,0.117647,0.035800,0.773810,0.971429,0.750
1,0.935873,0.794118,0.868735,0.690476,0.928571,0.750
2,0.975805,0.882353,0.949881,0.904762,0.700000,0.750
3,0.648524,0.558824,0.293556,0.666667,0.314286,0.875
4,0.574012,0.352941,0.396181,0.809524,0.400000,0.875


# 10. Apply the AHP Weights

In [49]:
weighted_matrix = normalized_decision.mul(
    priority_vector,
    axis=1
)

weighted_matrix.head()

,Project_Budget_USD,Estimated_Timeline_Months,Complexity_Score,Previous_Delivery_Success_Rate,Resource_Availability,Historical_Risk_Incidents
0,0.023295,0.008620,0.006911,0.325545,0.093783,0.134214
1,0.035093,0.058183,0.167698,0.290486,0.089646,0.134214
2,0.036591,0.064648,0.183362,0.380637,0.067579,0.134214
3,0.024318,0.040944,0.056667,0.280470,0.030342,0.156583
4,0.021524,0.025859,0.076477,0.340570,0.038617,0.156583


# 11. Calculate the Project Scores

In [50]:
scores = weighted_matrix.sum(axis=1)

scores.head()

0    0.592368
1    0.775320
2    0.867031
3    0.589323
4    0.659631
dtype: float64

# 12. Generate the Project Ranking

In [51]:
ranking = df.copy()

ranking["AHP_Score"] = scores

ranking = ranking.sort_values(
    by="AHP_Score",
    ascending=False
)

ranking.head(20)

,Project_ID,Project_Type,Team_Size,Project_Budget_USD,Estimated_Timeline_Months,Complexity_Score,Stakeholder_Count,Methodology_Used,Team_Experience_Level,Past_Similar_Projects,...,Client_Experience_Level,Change_Control_Maturity,Risk_Management_Maturity,Team_Colocation,Documentation_Quality,Project_Start_Month,Current_Phase_Duration_Months,Seasonal_Risk_Factor,Risk_Level,AHP_Score
613,PROJ_0614,Manufacturing,3,265992.55,4,1.91,3,Hybrid,Senior,5,...,First-time,Basic,Basic,Fully Remote,Good,11,1,1.0,High,0.948052
3496,PROJ_3497,Manufacturing,9,388789.62,5,2.44,8,Waterfall,Senior,5,...,First-time,Basic,NaN,Fully Remote,Basic,8,1,1.0,Medium,0.937929
1685,PROJ_1686,Healthcare,2,428345.18,4,1.89,7,Scrum,Junior,1,...,First-time,Formal,Basic,Hybrid,Excellent,8,1,1.0,Medium,0.935575
939,PROJ_0940,Healthcare,4,491707.71,4,2.04,6,Scrum,Mixed,2,...,Occasional,Formal,NaN,Fully Colocated,Good,12,1,1.0,Low,0.932549
2030,PROJ_2031,Manufacturing,2,310468.99,3,1.67,4,Kanban,Junior,1,...,Occasional,Formal,Advanced,Fully Remote,Basic,7,1,1.0,Low,0.930085
3981,PROJ_3982,Manufacturing,9,623104.86,8,3.02,7,Hybrid,Mixed,0,...,Regular,Basic,Basic,Hybrid,Good,8,2,1.0,High,0.920823
2844,PROJ_2845,Manufacturing,8,647088.67,6,2.65,10,Kanban,Mixed,3,...,Occasional,Formal,Formal,Fully Remote,Good,8,1,1.0,Medium,0.918382
1069,PROJ_1070,Manufacturing,6,331644.10,8,3.05,3,Hybrid,Mixed,1,...,First-time,Formal,NaN,Partially Colocated,Excellent,3,2,1.0,High,0.914523
2118,PROJ_2119,Healthcare,5,287879.33,6,2.11,6,Scrum,Expert,4,...,First-time,Advanced,Advanced,Partially Colocated,Poor,4,1,1.0,Low,0.913478
2067,PROJ_2068,Manufacturing,7,591695.70,7,1.64,4,Hybrid,Junior,2,...,Regular,Formal,Basic,Fully Colocated,Basic,12,2,1.0,Low,0.913064
